# SCP01 — Phase 2b EfficientNetB0 (HAM10000, mel-recall focus)

**GitHub:** [Ashu863804/SCP](https://github.com/Ashu863804/SCP)  
**Kaggle notebook:** SCP01

Phase 2b improvements: lesion-grouped split, train oversampling, clinical class weights, stronger aug, best-checkpoint eval, TTA.

**First run after this update:** set `FORCE_REORGANIZE = True` in the config cell (then you may set it back to `False`).

## 1. Setup & imports

In [ ]:
# --- 1) Clone GitHub repo (always — no if checks) ---
# Repo: https://github.com/Ashu863804/SCP
!rm -rf /kaggle/working/SCP
!git clone https://github.com/Ashu863804/SCP.git /kaggle/working/SCP

# --- 2) Path bootstrap (must run BEFORE any `import src...`) ---
import os
import sys
from pathlib import Path

GITHUB_USER = "Ashu863804"
REPO_NAME = "SCP"
REPO_DIR = Path(f"/kaggle/working/{REPO_NAME}")

def _bootstrap_project_path() -> Path:
    """Locate folder with src/config.py and add it to sys.path."""
    candidates = [
        REPO_DIR,
        REPO_DIR / "skin-cancer-detection",
        Path("/kaggle/working"),
        Path.cwd(),
        Path.cwd().parent,
        *list(Path.cwd().parents)[:6],
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        candidates.extend(sorted(kaggle_input.iterdir(), key=lambda p: p.name))

    seen = set()
    for root in candidates:
        try:
            root = root.resolve()
        except OSError:
            continue
        if root in seen:
            continue
        seen.add(root)
        if (root / "src" / "config.py").is_file():
            root_str = str(root)
            if root_str not in sys.path:
                sys.path.insert(0, root_str)
            os.chdir(root)
            return root

    raise FileNotFoundError(
        f"src/config.py not found after cloning https://github.com/{GITHUB_USER}/{REPO_NAME}. "
        "Ensure the GitHub repo contains the src/ folder at its root."
    )

PROJECT_ROOT = _bootstrap_project_path()

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src" / "config.py").exists())

## 2. Configuration

In [ ]:
from src.config import get_config

config = get_config()

# --- REQUIRED once after pulling Phase 2b code ---
FORCE_REORGANIZE = True   # rebuild train/val/test folders; then set False for later runs

# Optional overrides (defaults already tuned for mel / akiec / bcc recall):
# config.clinical_boost_factor = 1.5
# config.oversample_target_ratio = 0.5
# config.use_tta = True
# config.kaggle_dataset_slug = "datasets/kmader/skin-cancer-mnist-ham10000"
# config.__post_init__()

print("Phase 2b settings:")
print("  lesion-grouped split:", config.use_lesion_grouped_split)
print("  oversample train:", config.oversample_minority_train)
print("  clinical boost:", config.clinical_boost_classes, "x", config.clinical_boost_factor)
print("  TTA:", config.use_tta)
print("Kaggle input:", config.kaggle_input_dir)
print("Organized data:", config.organized_dir)
print("Outputs:", config.output_dir)

## 3. Dataset loading

In [ ]:
from src.dataset import prepare_dataset_pipeline

data = prepare_dataset_pipeline(config, force_organize=FORCE_REORGANIZE)

train_generator = data["train_generator"]
val_generator = data["val_generator"]
test_generator = data["test_generator"]
test_generator_tta = data["test_generator_tta"]
class_weight_dict = data["class_weight_dict"]
CLASS_NAMES = data["class_names"]
NUM_CLASSES = data["num_classes"]

print("Classes:", CLASS_NAMES)
print("Clinical boosted weights:", data["clinical_weights"])

## 4. Model creation

In [ ]:
from src.model import create_and_compile_model

model, base_model = create_and_compile_model(NUM_CLASSES, config)
model.summary()

## 5. Training (frozen backbone)

In [ ]:
from src.train import get_callbacks, train_model

callbacks = get_callbacks(config)
history = train_model(
    model,
    train_generator,
    val_generator,
    config,
    class_weight_dict=class_weight_dict,
    callbacks=callbacks,
)

## 6. Fine-tuning (top layers — same as original notebook)

In [ ]:
from src.train import fine_tune_model

history_fine = fine_tune_model(
    model,
    base_model,
    train_generator,
    val_generator,
    config,
    class_weight_dict=class_weight_dict,
    callbacks=callbacks,
)

## 7. Evaluation

In [ ]:
from src.evaluate import evaluate_best_checkpoint, save_evaluation_artifacts
from src.train import save_final_from_best

# Evaluate the best checkpoint (not the last epoch) + optional TTA
results = evaluate_best_checkpoint(
    test_generator,
    CLASS_NAMES,
    config,
    test_generator_tta=test_generator_tta,
)
save_evaluation_artifacts(results, config)
save_final_from_best(config)

print(f"Accuracy:          {results['accuracy']:.4f}")
print(f"Balanced accuracy: {results['balanced_accuracy']:.4f}")
print(f"Macro F1:          {results['macro_f1']:.4f}")

## 8. Plots & visualizations

In [ ]:
from src.evaluate import load_best_model
from src.utils import (
    merge_histories,
    plot_confusion_matrix,
    plot_sample_predictions,
    plot_training_history,
)

eval_model = load_best_model(config)
merged = merge_histories(history, history_fine)
plot_training_history(merged, save_path=config.history_plot_path)

plot_confusion_matrix(
    results["y_true"],
    results["y_pred"],
    CLASS_NAMES,
    save_path=config.confusion_matrix_path,
    normalize=False,
)
plot_confusion_matrix(
    results["y_true"],
    results["y_pred"],
    CLASS_NAMES,
    save_path=config.confusion_matrix_norm_path,
    normalize=True,
    title="Confusion Matrix (normalized)",
)

images, labels = next(test_generator)
preds = eval_model.predict(images)
plot_sample_predictions(images, labels, preds, CLASS_NAMES)